# LAB S3. Static and contextual word embeddings for text representation
### Natural Language and Information Retrieval
### Degree in Data Science · ETSINF

This notebook solves **Lab Session 3** using the corpus `EXIST2026_training_subset.json`.

We build five text representations:
- three **static** embedding-based document representations,
- two **contextual** embedding-based document representations.

For each representation, we identify the **most similar pair of texts** inside each class:
- `SEXIST`
- `NON-SEXIST`

using **cosine similarity**.

# Static embeddings

Let the corpus be

$$
\mathcal{D} = \{d_1, d_2, \dots, d_N\}
$$

and let each text belong to one class

$$
y_i \in \{	ext{SEXIST}, 	ext{NON-SEXIST}\}.
$$

For a static embedding model, every word $w$ is mapped to a fixed vector

$$
\mathbf{e}(w) \in \mathbb{R}^m.
$$

A document representation is obtained by averaging the embeddings of the words that remain after preprocessing:

$$
\mathbf{x}_i =

rac{1}{|W_i|} \sum_{w \in W_i} \mathbf{e}(w),
$$

where $W_i$ is the set of valid tokens of document $d_i$ that are found in the embedding model vocabulary.

## Some libraries

In [5]:
import fasttext.util
import gensim.downloader as api
from gensim.models.keyedvectors import KeyedVectors
import nltk
from nltk.tokenize import word_tokenize, TweetTokenizer
import numpy as np
import pandas as pd
import re
import json
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/sortmon/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/sortmon/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Load dataset and create the final label

The field `labels_task2_1` contains six annotations (`YES` or `NO`) for each meme. We compute the final class by **majority voting**:

$$
Y_i = \sum_{j=1}^{6} \mathbf{1}(a_{ij}=	ext{YES}), \qquad
N_i = \sum_{j=1}^{6} \mathbf{1}(a_{ij}=	ext{NO}).
$$

Then,

$$
	ext{label}(d_i)=\begin{cases}
	ext{SEXIST}, & 	ext{if } Y_i \ge 4 \
	ext{NON-SEXIST}, & 	ext{if } N_i \ge 4
\end{cases}
$$

In [6]:
import json
import pandas as pd

filename = "EXIST2026_training_subset.json"

with open(filename, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

rows = []
for _, item in raw_data.items():
    votes = item["labels_task2_1"]
    yes_votes = sum(v == "YES" for v in votes)
    no_votes = sum(v == "NO" for v in votes)

    if yes_votes >= 4:
        final_label = "SEXIST"
    elif no_votes >= 4:
        final_label = "NON-SEXIST"
    else:
        final_label = "TIE"  # Should not happen in this subset

    rows.append({
        "id_EXIST": item["id_EXIST"],
        "text": item["text"],
        "labels_task2_1": votes,
        "yes_votes": yes_votes,
        "no_votes": no_votes,
        "label": final_label,
    })

# The activity states that this subset only contains English texts.
df = pd.DataFrame(rows)
df = df[df["label"] != "TIE"].reset_index(drop=True)

print("Dataset size:", len(df))
print(df["label"].value_counts())
df.head()

Dataset size: 1000
label
SEXIST        553
NON-SEXIST    447
Name: count, dtype: int64


,id_EXIST,text,labels_task2_1,yes_votes,no_votes,label
0,210087,LADIES PLEASE CONTAIN YOUR ORGASM,"[NO, NO, NO, NO, NO, YES]",1,5,NON-SEXIST
1,210243,Me seeing boobs (age 15) Me seeing boobs (age ...,"[YES, NO, YES, NO, YES, YES]",4,2,SEXIST
2,210530,WWIII *begins* Feminists:,"[YES, NO, YES, YES, YES, YES]",5,1,SEXIST
3,211995,men should get vasectomies as a form of birth ...,"[NO, NO, NO, NO, NO, YES]",1,5,NON-SEXIST
4,211239,"WOMEN AREN'T OBJECTS? FALSE, AN OBJECT IS ANYT...","[YES, YES, YES, NO, YES, YES]",5,1,SEXIST


### Interpretation of the dataset loading output

The corpus contains **1000 English memes**, and the majority-voting step produces the following class distribution:

$$
N = 1000, \qquad |\mathcal{D}_{\text{SEXIST}}| = 553, \qquad |\mathcal{D}_{\text{NON-SEXIST}}| = 447
$$

This means the dataset is **slightly imbalanced toward the SEXIST class**, although the imbalance is not extreme. This is important because later we compute similarity **inside each class separately**, so both partitions still contain enough texts to obtain meaningful nearest-pair results.

The preview of the dataframe also confirms that:
- each row stores one meme,
- the original text is preserved,
- and a final label has already been assigned after aggregating the six annotations.


## Preprocess and tokenize the corpus for static embeddings

For the **static embedding** models, the activity requires the following preprocessing:
- remove URLs,
- remove hashtags,
- remove user references,
- lowercase,
- tokenize using `word_tokenize`,
- remove English stopwords. fileciteturn3file0

If $T_i$ is the original text, we define the cleaned text as

$$
T_i' = 	ext{tokenize}(	ext{lower}(	ext{removeUsers}(	ext{removeHashtags}(	ext{removeURLs}(T_i))))).
$$

In [8]:
# Info: nltk.tokenize.word_tokenize(text, language='english', preserve_line=False)
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download("punkt")
nltk.download("stopwords")

stopw = {
    "english": set(stopwords.words("english")),
    "spanish": set(stopwords.words("spanish")),
}

url_re = re.compile(r"https?://\S+|www\.\S+")
hashtag_re = re.compile(r"#\w+")
user_re = re.compile(r"@\w+")


def preprocess(text):
    text = url_re.sub(" ", text)
    text = hashtag_re.sub(" ", text)
    text = user_re.sub(" ", text)
    text = text.lower().strip()
    return text


def tokenize(text_list, lang="english"):
    result = []
    stop_words = stopw[lang]

    for text in text_list:
        clean_text = preprocess(text)
        tokens = word_tokenize(clean_text, language=lang)
        tokens = [tok for tok in tokens if tok.isalpha()]
        tokens = [tok for tok in tokens if tok not in stop_words]
        result.append(tokens)

    return result


tokenized_text = {
    "english": tokenize(df["text"].tolist(), "english"),
}

print(df["text"].head(3))
print("Sample tokenized texts:")
for tokens in tokenized_text["english"][:5]:
    print(tokens)

[nltk_data] Downloading package punkt to /home/sortmon/nltk_data...


0                  LADIES PLEASE CONTAIN YOUR ORGASM  
1    Me seeing boobs (age 15) Me seeing boobs (age ...
2                          WWIII *begins* Feminists:  
Name: text, dtype: str
Sample tokenized texts:
['ladies', 'please', 'contain', 'orgasm']
['seeing', 'boobs', 'age', 'seeing', 'boobs', 'age', 'de', 'mematic']
['wwiii', 'begins', 'feminists']
['men', 'get', 'vasectomies', 'form', 'birth', 'control', 'change', 'mind']
['women', 'objects', 'false', 'object', 'anything', 'mass', 'takes', 'space']


[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/sortmon/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Interpretation of the preprocessing output

The sample output shows that the preprocessing pipeline for **static embeddings** is working as intended. The original text is transformed into a cleaned token sequence where the most content-bearing words remain, for example:

$$
T_i \rightarrow W_i' = (w_{i1}, w_{i2}, \dots, w_{im_i})
$$

In practice, this means:
- uppercase words are normalized to lowercase,
- punctuation/noise is reduced,
- stopwords are removed,
- and the final token list is more semantically focused.

This step is important because static document embeddings are computed by **averaging word vectors**. Therefore, noisy tokens or very frequent functional words would otherwise distort the average:

$$
\mathbf{x}_i = \frac{1}{m_i} \sum_{k=1}^{m_i} \mathbf{e}(w_{ik})
$$

So the quality of this preprocessing directly affects the quality of the final static text representation.


## Text representation using static embeddings

We use the following pre-trained English models required by the activity:
- `word2vec-google-news-300`
- `fasttext-wiki-news-subwords-300`
- `glove-wiki-gigaword-300` fileciteturn3file0

For each model, each text is represented by the **average embedding** of its valid tokens.

### Load the models for English: word2vec, fasttext and glove

In [9]:
# Download / load the required pre-trained English embedding models from Gensim
import gensim.downloader as api

static_models = {
    "word2vec-google-news-300": api.load("word2vec-google-news-300"),
    "fasttext-wiki-news-subwords-300": api.load("fasttext-wiki-news-subwords-300"),
    "glove-wiki-gigaword-300": api.load("glove-wiki-gigaword-300"),
}

for name, model in static_models.items():
    print(name, model.vector_size)

word2vec-google-news-300 300
fasttext-wiki-news-subwords-300 300
glove-wiki-gigaword-300 300


### Interpretation of the static model loading output

All three required pre-trained static models were loaded successfully:

- `word2vec-google-news-300`
- `fasttext-wiki-news-subwords-300`
- `glove-wiki-gigaword-300`

The output shows that **all of them use 300-dimensional embeddings**:

$$
\mathbf{e}(w) \in \mathbb{R}^{300}
$$

This is useful because it makes the comparison between the three static approaches cleaner: the dimensionality is the same, so the differences in the results come mainly from the **embedding method itself** rather than from the vector size.

Conceptually:
- **Word2Vec** captures distributional similarity from local contexts.
- **FastText** also uses subword information, so it is more robust to rare or morphologically unusual words.
- **GloVe** relies on global co-occurrence statistics.


### Compute document representations from static embeddings

If a document token sequence is

$$
W_i = (w_{i1}, w_{i2}, \dots, w_{in_i}),
$$

its document vector is computed as

$$
\mathbf{x}_i^{(s)} = \frac{1}{M_i} \sum_{k=1}^{n_i} \mathbf{e}_s(w_{ik}) \cdot \mathbf{1}(w_{ik} \in V_s),
$$

where:
- $s$ denotes the static embedding model,
- $V_s$ is the vocabulary covered by model $s$,
- $M_i$ is the number of valid tokens found in that vocabulary.

If a text has no valid tokens after preprocessing, we return the zero vector.

In [10]:
import numpy as np


def gensim_sentence_rep(words, keyedvec):
    """
    Compute the word-embedding representation of a list of tokens by averaging
    the vectors of all tokens found in the model vocabulary.
    """
    dim = keyedvec.vector_size
    avg_vec = np.zeros(dim, dtype=float)
    total_w = 0

    for w in words:
        if w in keyedvec:
            avg_vec += keyedvec[w]
            total_w += 1

    if total_w == 0:
        return np.zeros(dim, dtype=float)

    return avg_vec / total_w


static_representations = {}
for model_name, keyedvec in static_models.items():
    X = np.vstack([
        gensim_sentence_rep(tokens, keyedvec)
        for tokens in tokenized_text["english"]
    ])
    static_representations[model_name] = X
    print(model_name, X.shape)

word2vec-google-news-300 (1000, 300)
fasttext-wiki-news-subwords-300 (1000, 300)
glove-wiki-gigaword-300 (1000, 300)


### Interpretation of the static text representation shapes

The output

$$
(1000, 300)
$$

for each static model means that we successfully built **one dense 300-dimensional vector per text** for the whole dataset:

$$
X^{(r)} \in \mathbb{R}^{1000 \times 300}
$$

where \(r\) is one of:
- Word2Vec,
- FastText,
- GloVe.

This confirms that the averaging procedure did not fail globally and that every text now has a fixed-size representation. This is the key difference with the representations from Lab 2: instead of sparse lexical vectors, we now work with **dense semantic vectors**.


## Compute cosine similarities for static embeddings

Given two document vectors $\mathbf{x}_i$ and $\mathbf{x}_j$, we use cosine similarity

For each representation, we compute the most similar pair **inside each class separately**.

In [11]:
from sklearn.metrics.pairwise import cosine_similarity


def most_similar_pair(X, sub_df):
    idx = sub_df.index.to_numpy()
    X_sub = X[idx]
    S = cosine_similarity(X_sub)
    np.fill_diagonal(S, -np.inf)
    i, j = np.unravel_index(np.argmax(S), S.shape)

    return {
        "id1": sub_df.iloc[i]["id_EXIST"],
        "text1": sub_df.iloc[i]["text"],
        "id2": sub_df.iloc[j]["id_EXIST"],
        "text2": sub_df.iloc[j]["text"],
        "similarity": float(S[i, j]),
    }


static_results = {}
for rep_name, X in static_representations.items():
    static_results[rep_name] = {
        "SEXIST": most_similar_pair(X, df[df["label"] == "SEXIST"]),
        "NON-SEXIST": most_similar_pair(X, df[df["label"] == "NON-SEXIST"]),
    }

## Show results

In [13]:
for rep_name, rep_results in static_results.items():
    print("=" * 80)
    print(rep_name)
    for cls, info in rep_results.items():
        print(f"Class: {cls}")
        print(f"ID1: {info['id1']}")
        print(f"Text1: {info['text1']}")
        print(f"ID2: {info['id2']}")
        print(f"Text2: {info['text2']}")
        print(f"Cosine similarity: {info['similarity']:.4f}")

word2vec-google-news-300
Class: SEXIST
ID1: 211469
Text1: Netflix: are you still watching? Somebody's daughter:  
ID2: 211199
Text2: Netflix: Are you still watching? Someone's Daughter: FAILGAGS.COM  
Cosine similarity: 0.9878
Class: NON-SEXIST
ID1: 211296
Text1: Feminists don't hate men they hate the patriarchy- which is a system that oppresses both men and women. MEMES  
ID2: 210822
Text2: I HATE MEN THATE WOMEN I HATE HUMANS G THAT'S MISANDRY THAT'S MISOGYNY  
Cosine similarity: 0.8684
fasttext-wiki-news-subwords-300
Class: SEXIST
ID1: 211469
Text1: Netflix: are you still watching? Somebody's daughter:  
ID2: 211199
Text2: Netflix: Are you still watching? Someone's Daughter: FAILGAGS.COM  
Cosine similarity: 0.9954
Class: NON-SEXIST
ID1: 211935
Text1: WOMEN ON SEX STRIKE MEN MARVIN makeameme.org  
ID2: 211932
Text2: Men: "we have to do all the work during sex, women have it easy" Women: 8  
Cosine similarity: 0.9400
glove-wiki-gigaword-300
Class: SEXIST
ID1: 211469
Text1: Netflix: a

### Analysis of the static embedding results

A very clear pattern appears in the **SEXIST** class: the three static models return the **same pair** as the most similar one:

- `211469`: *"Netflix: are you still watching? Somebody's daughter:"*
- `211199`: *"Netflix: Are you still watching? Someone's Daughter: FAILGAGS.COM"*

This is a strong sign that the pair is not just semantically related, but also **almost template-identical**. The cosine values are all extremely high, especially for FastText:

$$
\cos_{\text{w2v}} = 0.9878, \qquad
\cos_{\text{ft}} = 0.9954, \qquad
\cos_{\text{glove}} = 0.9855
$$

FastText gives the highest value, which is consistent with its use of **subword information**. Small lexical differences such as *somebody* vs. *someone's* can still lead to very close vectors.

For the **NON-SEXIST** class, the most similar pair changes depending on the model. This is also informative: unlike the SEXIST case, the nearest pair is less universally obvious and depends more on the inductive bias of each embedding space. In other words, the three static methods agree strongly when the texts are near-duplicates, but they may diverge when similarity is more semantic and less template-driven.


# Contextual embeddings

Unlike static embeddings, contextual models produce token representations that depend on the full sentence context.

If a transformer processes a tokenized sequence and returns hidden vectors

$$
\mathbf{h}_1, \mathbf{h}_2, \dots, \mathbf{h}_L,
$$

we represent each text by the output vector associated with the special classification token:

$$
\mathbf{x}_i^{(c)} = \mathbf{h}_{	ext{[CLS]}}
$$

for BERT-like architectures. In practice, this corresponds to the first token representation in the final hidden layer.

## Some libraries

In [15]:
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from transformers import BertTokenizer, BertModel, RobertaTokenizer, RobertaModel
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

## Contextual models required by the activity

We use the two English transformer models required in the PDF:
- `bert-base-uncased`
- `roberta-base` fileciteturn3file0

The activity explicitly states that **no preprocessing is required** for transformer-based models. We therefore feed the raw text directly to the tokenizer and the model. fileciteturn3file0

In [16]:
modelnames = {
    "english": ["bert-base-uncased", "roberta-base"],
   # "spanish": ["dccuchile/bert-base-spanish-wwm-uncased"]
}

## Which device to use?

In [17]:
if torch.backends.mps.is_available():  # Mac M? GPU
    device = torch.device("mps")
elif torch.cuda.is_available():  # Nvidia GPU
    device = torch.device("cuda")
else:  # CPU
    device = torch.device("cpu")
print(device)

cuda


### Interpretation of the device selection output

The output shows that the notebook is using:

$$
\texttt{device} = \texttt{cuda}
$$

This means the contextual models are running on an **NVIDIA GPU**, which is highly beneficial for transformer inference. This matters because BERT and RoBERTa are much heavier than the static embedding models, and processing 1000 texts in batches on CPU would be significantly slower.

So this output is not only technical; it confirms that the contextual part of the practice is being executed under a computationally appropriate setup.


## Load the tokenizers and the models

In [18]:
# Load the contextual tokenizers and models with the generic Hugging Face classes
contextual_tokenizers = {}
contextual_models = {}

for model_name in modelnames["english"]:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.eval()
    model.to(device)

    contextual_tokenizers[model_name] = tokenizer
    contextual_models[model_name] = model

    print(f"Loaded: {model_name}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded: bert-base-uncased


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded: roberta-base


### Interpretation of the contextual model loading output

This cell loads the tokenizers and the base transformer encoders for:
- `bert-base-uncased`
- `roberta-base`

The long loading reports and progress bars are expected, since the models are downloaded and their weights are initialized.

A useful detail is the BERT load report mentioning **unexpected `cls.seq_relationship.*` weights**. This is **not an error** in our use case. We are loading `BertModel`, that is, the encoder backbone only, and we do **not** need the next-sentence-prediction head. Therefore, those extra weights are irrelevant for this activity. What matters is that the encoder itself loads correctly, because we only need the hidden states to extract the `[CLS]` representation.


## Compute document representations with contextual embeddings

Because the full dataset is large, the PDF recommends processing the texts in **batches**. fileciteturn3file0

For each batch, we:
1. tokenize the raw texts,
2. pad and truncate them,
3. feed them to the transformer,
4. extract the contextual vector of the first token,
5. concatenate all batches.

In [19]:
batch_size = 16
raw_texts = df["text"].tolist()


def contextual_representations(texts, tokenizer, model, batch_size=16, max_length=128):
    all_cls = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = model(**encoded)
            last_hidden_state = outputs.last_hidden_state
            cls_batch = last_hidden_state[:, 0, :]
            all_cls.append(cls_batch.cpu())

    return torch.cat(all_cls, dim=0).numpy()


contextual_reps = {}
for model_name in modelnames["english"]:
    tokenizer = contextual_tokenizers[model_name]
    model = contextual_models[model_name]
    X = contextual_representations(raw_texts, tokenizer, model, batch_size=batch_size, max_length=128)
    contextual_reps[model_name] = X
    print(model_name, X.shape)

bert-base-uncased (1000, 768)
roberta-base (1000, 768)


### Interpretation of the contextual representation shapes

The output shows:

$$
X^{(\text{bert})} \in \mathbb{R}^{1000 \times 768}, \qquad
X^{(\text{roberta})} \in \mathbb{R}^{1000 \times 768}
$$

This confirms that:
1. the full dataset was processed successfully in batches,
2. the `[CLS]` representation was extracted for every text,
3. and both contextual models return a **768-dimensional document vector**.

So at this point each meme has two contextual representations, one from BERT and one from RoBERTa, ready for similarity analysis.


## Compute cosine similarities

In [20]:
contextual_results = {}
for rep_name, X in contextual_reps.items():
    contextual_results[rep_name] = {
        "SEXIST": most_similar_pair(X, df[df["label"] == "SEXIST"]),
        "NON-SEXIST": most_similar_pair(X, df[df["label"] == "NON-SEXIST"]),
    }

## Show results

In [21]:
for rep_name, rep_results in contextual_results.items():
    print("=" * 80)
    print(rep_name)
    for cls, info in rep_results.items():
        print(f"Class: {cls}")
        print(f"ID1: {info['id1']}")
        print(f"Text1: {info['text1']}")
        print(f"ID2: {info['id2']}")
        print(f"Text2: {info['text2']}")
        print(f"Cosine similarity: {info['similarity']:.4f}")

bert-base-uncased
Class: SEXIST
ID1: 210096
Text1: When she actually puts effort once during sex right before leaving you MEMES  
ID2: 211100
Text2: When the dick is just too good MEMES  
Cosine similarity: 0.9674
Class: NON-SEXIST
ID1: 211684
Text1: REVERSE STEREOTYPE Think about which one of these two has done time ICANHASCHEEZEURGER.COM  
ID2: 210116
Text2: WHAT YOUR NON BINARY GIRLFRIEND THINK'S: DAMN GIRL YOU THICCCCCC makeameme.org  
Cosine similarity: 0.9639
roberta-base
Class: SEXIST
ID1: 211858
Text1: RIHANNA? COCKSUCKER WHORE makeameme.org  
ID2: 211551
Text2: CUM SLUT WHORE makeameme.org  
Cosine similarity: 0.9996
Class: NON-SEXIST
ID1: 211126
Text1: GET NAKED IT'S FRIDAY! makeameme.org  
ID2: 211108
Text2: I'M NAKED WAIT IN NAKED? AHHHHH! makeameme.org  
Cosine similarity: 0.9994


### Analysis of the contextual embedding results

The contextual models produce **very high cosine similarities**, especially for short meme-like texts built around strong templates or repeated lexical patterns.

For **BERT**, the top SEXIST pair is:

- `210096`: *"When she actually puts effort once during sex right before leaving you MEMES"*
- `211100`: *"When the dick is just too good MEMES"*

with

$$
\cos_{\text{BERT}} = 0.9674
$$

For **RoBERTa**, the top SEXIST pair is:

- `211858`: *"RIHANNA? COCKSUCKER WHORE makeameme.org"*
- `211551`: *"CUM SLUT WHORE makeameme.org"*

with

$$
\cos_{\text{RoBERTa}} = 0.9996
$$

and the NON-SEXIST pair also reaches a value close to 1:

$$
\cos_{\text{RoBERTa, NON}} = 0.9994
$$

This suggests two things:

1. **Contextual models are very effective at collapsing very short, strongly formulaic meme texts into nearby regions of the embedding space.**
2. **RoBERTa seems especially sensitive to short, highly patterned texts**, which may explain why some cosine values are almost maximal.

Compared with the static models, the contextual models are not averaging isolated word vectors. Instead, they encode the full sentence jointly and then summarize it through the `[CLS]` vector. That is why they can produce extremely tight clustering even when the texts are not word-for-word identical.
